In [ ]:
import socket

print(socket.gethostname())

midway3-0013.rcc.local


In [ ]:
import pandas as pd
import csv
from pathlib import Path

base = Path(
    "/project/rcc/users/hdashti/projects/shiplogs/oldweather/Cleaned_L2_Spreadsheets/Cleaned L2 Spreadsheets"
)

COL_MAP = {
    # Position & time
    "YYYYMMDD": "date",
    "Hour": "hour",
    "Lat": "lat",
    "Long": "lon",
    "Course": "course",
    " Course": "course",
    "Distance": "distance",
    # Wind
    "DirM": "wind_dir_mag",
    "DirT": "wind_dir_true",
    "Dir": "wind_dir_mag",
    "Kts": "wind_kts",
    # Pressure
    "Baro": "baro",
    # Temperature
    "Dry": "temp_dry",
    "Wet": "temp_wet",
    "Water": "temp_water",
    "SeaT": "temp_water",
    "Sea": "temp_water",
    # Sky & weather
    "Weather": "weather",
    "Clouds": "clouds",
    "Amount": "cloud_amount",
    "Clear": "clear_sky",
    # Metadata & OCR reference
    "Note": "note",
    "URL_W": "url_w",
    # Original transcribed values
    "Orig Lat": "orig_lat",
    "Orig_Lat": "orig_lat",
    "OrigLat": "orig_lat",
    "jOrig Lat": "orig_lat",
    "Orig Lon": "orig_lon",
    "Orig_Lon": "orig_lon",
    "OrigLon": "orig_lon",
    "Orig Long": "orig_lon",
    "Orig Course": "orig_course",
    "Orig_Course": "orig_course",
    "OrigCourse": "orig_course",
    " Orig Course": "orig_course",
}

KEEP_COLS = list(set(COL_MAP.values())) + ["ship", "source_file"]

In [ ]:
all_dfs = []

for ship_dir in sorted(base.iterdir()):
    if not ship_dir.is_dir():
        continue
    ship_name = ship_dir.name
    for f in sorted(ship_dir.glob("*.tsv")):
        try:
            df = pd.read_csv(
                f,
                sep="\t",
                dtype=str,
                encoding="utf-8",
                encoding_errors="replace",
                quoting=csv.QUOTE_NONE,
            )
            df = df.loc[:, ~df.columns.duplicated()]
            rename = {c: COL_MAP[c] for c in df.columns if c in COL_MAP}
            df.rename(columns=rename, inplace=True)
            df = df.loc[:, ~df.columns.duplicated()]
            df = df[[c for c in KEEP_COLS if c in df.columns]]
            df["ship"] = ship_name
            df["source_file"] = f.name
            all_dfs.append(df)
        except Exception as e:
            print(f"ERROR {f.name}: {e}")

df_all = pd.concat(all_dfs, ignore_index=True)
print(
    f"Loaded: {len(all_dfs)} files, {len(df_all):,} rows, {df_all['ship'].nunique()} ships"
)

Loaded: 356 files, 2,051,185 rows, 42 ships


In [ ]:
# === Old Weather Dataset Summary ===
print(f"{'=' * 50}")
print(f"OLD WEATHER MERGED DATASET")
print(f"{'=' * 50}")
print(f"Ships:          {df_all['ship'].nunique()}")
print(f"Files:          {df_all['source_file'].nunique()}")
print(f"Total rows:     {len(df_all):,}")
print(
    f"Date range:     {df_all['date'].dropna().replace('', pd.NA).dropna().min()} to {df_all['date'].dropna().replace('', pd.NA).dropna().max()}"
)
print(f"")

at_anchor = (df_all["distance"] == "a").sum()
uw = len(df_all) - at_anchor
print(f"At anchor:      {at_anchor:,} ({100 * at_anchor / len(df_all):.1f}%)")
print(f"Underway:       {uw:,} ({100 * uw / len(df_all):.1f}%)")

print(f"\nWeather coverage (all / underway only):")
underway = df_all[df_all["distance"] != "a"]
for col, label in [
    ("baro", "Barometer"),
    ("wind_kts", "Wind speed"),
    ("wind_dir_true", "Wind dir"),
    ("temp_dry", "Air temp"),
    ("temp_wet", "Wet bulb"),
    ("temp_water", "Sea temp"),
    ("weather", "Weather code"),
    ("clouds", "Clouds"),
]:
    if col in df_all.columns:
        n_all = df_all[col].replace("", pd.NA).notna().sum()
        n_uw = underway[col].replace("", pd.NA).notna().sum()
        print(
            f"  {label:>14}: {n_all:>10,} ({100 * n_all / len(df_all):5.1f}%)  |  {n_uw:>8,} ({100 * n_uw / len(underway):5.1f}%)"
        )

print(f"\nTop 10 ships by underway rows:")
uw_counts = underway.groupby("ship").size().sort_values(ascending=False)
for ship, n in uw_counts.head(10).items():
    sst = (
        underway[underway["ship"] == ship]["temp_water"]
        .replace("", pd.NA)
        .notna()
        .sum()
    )
    print(f"  {ship:>20}: {n:>7,} rows, SST {100 * sst / n:.0f}%")

OLD WEATHER MERGED DATASET
Ships:          42
Files:          356
Total rows:     2,051,185
Date range:     1859-05-04 to 1955-12-31

At anchor:      1,404,485 (68.5%)
Underway:       646,700 (31.5%)

Weather coverage (all / underway only):
       Barometer:  1,249,453 ( 60.9%)  |   445,637 ( 68.9%)
      Wind speed:  1,386,354 ( 67.6%)  |   480,406 ( 74.3%)
        Wind dir:  1,372,934 ( 66.9%)  |   476,398 ( 73.7%)
        Air temp:  1,281,846 ( 62.5%)  |   457,171 ( 70.7%)
        Wet bulb:  1,040,413 ( 50.7%)  |   361,854 ( 56.0%)
        Sea temp:    480,047 ( 23.4%)  |   322,672 ( 49.9%)
    Weather code:  1,389,555 ( 67.7%)  |   480,038 ( 74.2%)
          Clouds:  1,031,322 ( 50.3%)  |   362,078 ( 56.0%)

Top 10 ships by underway rows:
                  Bear:  77,681 rows, SST 36%
                 Haida:  37,877 rows, SST 87%
             Kearsarge:  36,346 rows, SST 66%
             Tuscarora:  35,957 rows, SST 70%
                Storis:  35,929 rows, SST 43%
         Burton_I

In [ ]:
out_path = Path(
    "/project/rcc/users/hdashti/projects/shiplogs/oldweather/merged_raw.parquet"
)
df_all.to_parquet(out_path, index=False)
print(f"Saved: {out_path} ({out_path.stat().st_size / 1e6:.1f} MB)")

Saved: /project/rcc/users/hdashti/projects/shiplogs/oldweather/merged_raw.parquet (32.8 MB)
